In [1]:
import requests
import pandas as pd
import os
import json
import time

from dotenv import load_dotenv

load_dotenv("../.env")

TMDB_ACCESS_TOKEN = os.getenv("TMDB_ACCESS_TOKEN")

print("TMDB Token Loaded:", bool(TMDB_ACCESS_TOKEN))

TMDB Token Loaded: True


In [2]:
def tmdb_request(endpoint, params=None):
    
    url = f"https://api.themoviedb.org/3{endpoint}"
    
    headers = {
        "Authorization": f"Bearer {TMDB_ACCESS_TOKEN}",
        "accept": "application/json"
    }
    
    for attempt in range(3):
        
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=15
            )
            
            if response.status_code == 200:
                return response.json()
            
            print(
                f"API Error: {response.status_code} | "
                f"{endpoint}"
            )
            
        except requests.exceptions.RequestException as e:
            print(
                f"Attempt {attempt + 1}/3 failed: {type(e).__name__}"
            )
            
            time.sleep(2)
    
    return None

In [3]:
def collect_movie_ids(total_pages=25):
    
    movie_ids = set()
    
    for page in range(1, total_pages + 1):
        
        data = tmdb_request(
            "/discover/movie",
            {
                "language": "en-US",
                "sort_by": "popularity.desc",
                "vote_count.gte": 50,
                "page": page
            }
        )
        
        if data is None:
            print(f"❌ Page {page} failed")
            continue
        
        for movie in data.get("results", []):
            movie_ids.add(movie["id"])
        
        print(
            f"Page {page}/{total_pages} | "
            f"Unique movies: {len(movie_ids)}"
        )
        
        time.sleep(0.2)
    
    return list(movie_ids)

In [4]:
movie_ids = collect_movie_ids(25)

print("\nTotal unique movie IDs:", len(movie_ids))

Page 1/25 | Unique movies: 20
Page 2/25 | Unique movies: 40
Page 3/25 | Unique movies: 60
Page 4/25 | Unique movies: 80
Page 5/25 | Unique movies: 96
Page 6/25 | Unique movies: 116
Page 7/25 | Unique movies: 136
Page 8/25 | Unique movies: 150
Page 9/25 | Unique movies: 170
Page 10/25 | Unique movies: 189
Page 11/25 | Unique movies: 203
Page 12/25 | Unique movies: 217
Page 13/25 | Unique movies: 237
Page 14/25 | Unique movies: 257
Page 15/25 | Unique movies: 271
Page 16/25 | Unique movies: 286
Page 17/25 | Unique movies: 299
Page 18/25 | Unique movies: 314
Page 19/25 | Unique movies: 329
Page 20/25 | Unique movies: 346
Page 21/25 | Unique movies: 361
Page 22/25 | Unique movies: 377
Page 23/25 | Unique movies: 388
Page 24/25 | Unique movies: 405
Page 25/25 | Unique movies: 421

Total unique movie IDs: 421


In [5]:
with open(
    "../data/raw/movie_ids.json",
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        movie_ids,
        f,
        indent=2
    )

print("✅ Movie IDs saved.")

✅ Movie IDs saved.


In [6]:
def enrich_movie_large(movie_id):
    
    details = tmdb_request(
        f"/movie/{movie_id}",
        {"language": "en-US"}
    )
    
    if details is None:
        return None

    credits = tmdb_request(
        f"/movie/{movie_id}/credits",
        {"language": "en-US"}
    )
    
    keywords_data = tmdb_request(
        f"/movie/{movie_id}/keywords"
    )
    
    if credits is None:
        credits = {}
        
    if keywords_data is None:
        keywords_data = {}
    
    
    genres = [
        genre["name"]
        for genre in details.get("genres", [])
    ]
    
    cast = [
        person["name"]
        for person in credits.get("cast", [])[:10]
    ]
    
    directors = [
        person["name"]
        for person in credits.get("crew", [])
        if person.get("job") == "Director"
    ]
    
    keywords = [
        keyword["name"]
        for keyword in keywords_data.get("keywords", [])
    ]
    
    
    return {
        "id": details.get("id"),
        "title": details.get("title"),
        "overview": details.get("overview"),
        "genres": genres,
        "keywords": keywords,
        "cast": cast,
        "director": directors,
        "runtime": details.get("runtime"),
        "release_date": details.get("release_date"),
        "vote_average": details.get("vote_average"),
        "vote_count": details.get("vote_count"),
        "popularity": details.get("popularity"),
        "poster_path": details.get("poster_path"),
        "backdrop_path": details.get("backdrop_path"),
        "original_language": details.get("original_language"),
        "media_type": "movie"
    }

In [7]:
enriched_movies = []

for i, movie_id in enumerate(movie_ids, start=1):
    
    print(f"[{i}/{len(movie_ids)}] Processing ID: {movie_id}")
    
    try:
        
        movie = enrich_movie_large(movie_id)
        
        if movie is not None:
            enriched_movies.append(movie)
        
    except Exception as e:
        
        print(f"❌ Failed ID {movie_id}: {e}")
    
    
    # Save every 25 movies
    if i % 25 == 0:
        
        with open(
            "../data/raw/movies_checkpoint.json",
            "w",
            encoding="utf-8"
        ) as f:
            
            json.dump(
                enriched_movies,
                f,
                ensure_ascii=False,
                indent=2
            )
        
        print(
            f"💾 CHECKPOINT: "
            f"{len(enriched_movies)} movies saved"
        )
    
    time.sleep(0.3)

[1/421] Processing ID: 198663
[2/421] Processing ID: 13
[3/421] Processing ID: 2062
[4/421] Processing ID: 976912
[5/421] Processing ID: 1110034
[6/421] Processing ID: 22
[7/421] Processing ID: 24
[8/421] Processing ID: 313369
[9/421] Processing ID: 28
[10/421] Processing ID: 497698
[11/421] Processing ID: 122917
[12/421] Processing ID: 436270
[13/421] Processing ID: 1321008
[14/421] Processing ID: 1038392
[15/421] Processing ID: 58
[16/421] Processing ID: 62
[17/421] Processing ID: 911430
[18/421] Processing ID: 74
[19/421] Processing ID: 550988
[20/421] Processing ID: 426063
[21/421] Processing ID: 718930
[22/421] Processing ID: 866398
[23/421] Processing ID: 1308767
[24/421] Processing ID: 98
[25/421] Processing ID: 101
💾 CHECKPOINT: 25 movies saved
[26/421] Processing ID: 1368166
[27/421] Processing ID: 82023
[28/421] Processing ID: 103
[29/421] Processing ID: 106
[30/421] Processing ID: 1226863
[31/421] Processing ID: 335984
[32/421] Processing ID: 111
[33/421] Processing ID: 9503

In [8]:
print("Movies enriched:", len(enriched_movies))

Movies enriched: 421


In [9]:
enriched_movies_df = pd.DataFrame(enriched_movies)

print("Dataset shape:", enriched_movies_df.shape)

Dataset shape: (421, 16)


In [10]:
enriched_movies_df.to_json(
    "../data/processed/movies_421_enriched.json",
    orient="records",
    indent=2,
    force_ascii=False
)

enriched_movies_df.to_csv(
    "../data/processed/movies_421_enriched.csv",
    index=False
)

print("✅ Final movie dataset saved.")

✅ Final movie dataset saved.


In [11]:
def collect_tv_ids(total_pages=25):
    
    tv_ids = set()
    
    for page in range(1, total_pages + 1):
        
        data = tmdb_request(
            "/discover/tv",
            {
                "language": "en-US",
                "sort_by": "popularity.desc",
                "vote_count.gte": 50,
                "page": page
            }
        )
        
        if data is None:
            print(f"❌ Page {page} failed")
            continue
        
        for show in data.get("results", []):
            tv_ids.add(show["id"])
        
        print(
            f"Page {page}/{total_pages} | "
            f"Unique TV shows: {len(tv_ids)}"
        )
        
        time.sleep(0.2)
    
    return list(tv_ids)

In [12]:
tv_ids = collect_tv_ids(25)

print("\nTotal unique TV IDs:", len(tv_ids))

Page 1/25 | Unique TV shows: 20
Page 2/25 | Unique TV shows: 38
Page 3/25 | Unique TV shows: 58
Page 4/25 | Unique TV shows: 76
Page 5/25 | Unique TV shows: 95
Page 6/25 | Unique TV shows: 115
Page 7/25 | Unique TV shows: 130
Page 8/25 | Unique TV shows: 149
Page 9/25 | Unique TV shows: 167
Page 10/25 | Unique TV shows: 182
Page 11/25 | Unique TV shows: 201
Page 12/25 | Unique TV shows: 218
Page 13/25 | Unique TV shows: 234
Page 14/25 | Unique TV shows: 252
Page 15/25 | Unique TV shows: 271
Page 16/25 | Unique TV shows: 291
Page 17/25 | Unique TV shows: 310
Page 18/25 | Unique TV shows: 330
Page 19/25 | Unique TV shows: 342
Page 20/25 | Unique TV shows: 353
Page 21/25 | Unique TV shows: 365
Page 22/25 | Unique TV shows: 382
Page 23/25 | Unique TV shows: 402
Page 24/25 | Unique TV shows: 419
Page 25/25 | Unique TV shows: 439

Total unique TV IDs: 439


In [13]:
with open(
    "../data/raw/tv_ids.json",
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        tv_ids,
        f,
        indent=2
    )

print("✅ TV IDs saved.")

✅ TV IDs saved.


In [14]:
def enrich_tv_show(tv_id):
    
    # 1. TV details
    details = tmdb_request(
        f"/tv/{tv_id}",
        {"language": "en-US"}
    )
    
    if details is None:
        return None
    
    
    # 2. Credits
    credits = tmdb_request(
        f"/tv/{tv_id}/credits",
        {"language": "en-US"}
    )
    
    # 3. Keywords
    keywords_data = tmdb_request(
        f"/tv/{tv_id}/keywords"
    )
    
    if credits is None:
        credits = {}
    
    if keywords_data is None:
        keywords_data = {}
    
    
    # Genres
    genres = [
        genre["name"]
        for genre in details.get("genres", [])
    ]
    
    
    # Top 10 cast
    cast = [
        person["name"]
        for person in credits.get("cast", [])[:10]
    ]
    
    
    # TV creators
    creators = [
        person["name"]
        for person in details.get("created_by", [])
    ]
    
    
    # Keywords
    keywords = [
        keyword["name"]
        for keyword in keywords_data.get("results", [])
    ]
    
    
    return {
        "id": details.get("id"),
        "title": details.get("name"),
        "overview": details.get("overview"),
        "genres": genres,
        "keywords": keywords,
        "cast": cast,
        "director": creators,
        "runtime": (
            details.get("episode_run_time", [None])[0]
            if details.get("episode_run_time")
            else None
        ),
        "release_date": details.get("first_air_date"),
        "vote_average": details.get("vote_average"),
        "vote_count": details.get("vote_count"),
        "popularity": details.get("popularity"),
        "poster_path": details.get("poster_path"),
        "backdrop_path": details.get("backdrop_path"),
        "original_language": details.get("original_language"),
        "media_type": "tv",
        "number_of_seasons": details.get("number_of_seasons"),
        "number_of_episodes": details.get("number_of_episodes")
    }

In [15]:
enriched_tv = []

for i, tv_id in enumerate(tv_ids, start=1):
    
    print(f"[{i}/{len(tv_ids)}] Processing TV ID: {tv_id}")
    
    try:
        
        show = enrich_tv_show(tv_id)
        
        if show is not None:
            enriched_tv.append(show)
        
    except Exception as e:
        
        print(f"❌ Failed ID {tv_id}: {e}")
    
    
    # Checkpoint every 25
    if i % 25 == 0:
        
        with open(
            "../data/raw/tv_checkpoint.json",
            "w",
            encoding="utf-8"
        ) as f:
            
            json.dump(
                enriched_tv,
                f,
                ensure_ascii=False,
                indent=2
            )
        
        print(
            f"💾 CHECKPOINT: "
            f"{len(enriched_tv)} TV shows saved"
        )
    
    time.sleep(0.3)

[1/439] Processing TV ID: 200709
[2/439] Processing TV ID: 194583
[3/439] Processing TV ID: 32798
[4/439] Processing TV ID: 71712
[5/439] Processing TV ID: 36
[6/439] Processing TV ID: 290856
[7/439] Processing TV ID: 10283
[8/439] Processing TV ID: 157741
[9/439] Processing TV ID: 45
[10/439] Processing TV ID: 71728
[11/439] Processing TV ID: 157744
[12/439] Processing TV ID: 2098
[13/439] Processing TV ID: 52
[14/439] Processing TV ID: 116799
[15/439] Processing TV ID: 61511
[16/439] Processing TV ID: 2122
[17/439] Processing TV ID: 34891
[18/439] Processing TV ID: 78
[19/439] Processing TV ID: 4177
[20/439] Processing TV ID: 2129
[21/439] Processing TV ID: 45140
[22/439] Processing TV ID: 90
[23/439] Processing TV ID: 95
[24/439] Processing TV ID: 278624
[25/439] Processing TV ID: 4194
💾 CHECKPOINT: 25 TV shows saved
[26/439] Processing TV ID: 69740
[27/439] Processing TV ID: 71789
[28/439] Processing TV ID: 71790
[29/439] Processing TV ID: 2171
[30/439] Processing TV ID: 4229
[31/4

In [16]:
print("TV shows enriched:", len(enriched_tv))

TV shows enriched: 439


In [17]:
enriched_tv_df = pd.DataFrame(enriched_tv)

print("TV Dataset Shape:", enriched_tv_df.shape)

TV Dataset Shape: (439, 18)


In [18]:
catalogue_df = pd.concat(
    [
        enriched_movies_df,
        enriched_tv_df
    ],
    ignore_index=True
)

print("Final Catalogue Shape:", catalogue_df.shape)

Final Catalogue Shape: (860, 18)


In [19]:
catalogue_df["media_type"].value_counts()

media_type
tv       439
movie    421
Name: count, dtype: int64

In [20]:
catalogue_df[
    [
        "title",
        "media_type",
        "genres",
        "keywords",
        "cast",
        "director"
    ]
].head(10)

,title,media_type,genres,keywords,cast,director
0,The Maze Runner,movie,"[Action, Mystery, Science Fiction, Thriller]","[based on novel or book, escape, dystopia, maz...","[Dylan O'Brien, Kaya Scodelario, Thomas Brodie...",[Wes Ball]
1,Forrest Gump,movie,"[Comedy, Drama, Romance]","[new year's eve, vietnam war, vietnam veteran,...","[Tom Hanks, Robin Wright, Gary Sinise, Sally F...",[Robert Zemeckis]
2,Ratatouille,movie,"[Animation, Comedy, Family, Fantasy]","[work, sibling relationship, paris, france, ex...","[Patton Oswalt, Ian Holm, Lou Romano, Brian De...",[Brad Bird]
3,Graphic Desires,movie,"[Thriller, Crime]","[infidelity, cheating, killing, murder, softco...","[David Wayman, Sian Altman, May Kelly, Ocean M...",[Andy Edwards]
4,Kraken,movie,"[Horror, Action, Thriller]","[sea monster, norwegian folklore, cautionary]","[Sara Khorami, Mikkel Bratt Silset, Ingvild Ho...",[Pål Øie]
5,Pirates of the Caribbean: The Curse of the Bla...,movie,"[Adventure, Fantasy, Action]","[ship, blacksmith, gold, exotic island, govern...","[Johnny Depp, Geoffrey Rush, Orlando Bloom, Ke...",[Gore Verbinski]
6,Kill Bill: Vol. 1,movie,"[Action, Crime]","[martial arts, japan, kung fu, sword, yakuza, ...","[Uma Thurman, Lucy Liu, Vivica A. Fox, Daryl H...",[Quentin Tarantino]
7,La La Land,movie,"[Comedy, Drama, Romance]","[dancing, dance, jazz, melancholy, musical, am...","[Ryan Gosling, Emma Stone, John Legend, Rosema...",[Damien Chazelle]
8,Apocalypse Now,movie,"[Drama, War]","[guerrilla warfare, vietnam war, journalist, m...","[Martin Sheen, Marlon Brando, Frederic Forrest...",[Francis Ford Coppola]
9,Black Widow,movie,"[Action, Adventure, Science Fiction]","[assassin, hero, spy, kgb, based on comic, fem...","[Scarlett Johansson, Florence Pugh, Rachel Wei...",[Cate Shortland]


In [21]:
catalogue_df.to_json(
    "../data/processed/movie_tv_catalogue.json",
    orient="records",
    indent=2,
    force_ascii=False
)

catalogue_df.to_csv(
    "../data/processed/movie_tv_catalogue.csv",
    index=False
)

print("✅ Unified Movie + TV catalogue saved.")

✅ Unified Movie + TV catalogue saved.


In [22]:
catalogue_df["title"].duplicated().sum()

np.int64(9)

In [23]:
catalogue_df[
    [
        "overview",
        "genres",
        "keywords",
        "cast",
        "director"
    ]
].isna().sum()

overview    0
genres      0
keywords    0
cast        0
director    0
dtype: int64

In [24]:
catalogue_df["media_type"].value_counts()

media_type
tv       439
movie    421
Name: count, dtype: int64

In [25]:
catalogue_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  860 non-null    int64  
 1   title               860 non-null    str    
 2   overview            860 non-null    str    
 3   genres              860 non-null    object 
 4   keywords            860 non-null    object 
 5   cast                860 non-null    object 
 6   director            860 non-null    object 
 7   runtime             634 non-null    float64
 8   release_date        860 non-null    str    
 9   vote_average        860 non-null    float64
 10  vote_count          860 non-null    int64  
 11  popularity          860 non-null    float64
 12  poster_path         860 non-null    str    
 13  backdrop_path       860 non-null    str    
 14  original_language   860 non-null    str    
 15  media_type          860 non-null    str    
 16  number_of_seasons  

In [28]:
catalogue_df["title"].duplicated().sum()
catalogue_df[
    ["overview", "genres", "keywords", "cast", "director"]
].isna().sum()
catalogue_df["media_type"].value_counts()
catalogue_df.shape

(860, 18)

In [29]:
print("Duplicate titles:", catalogue_df["title"].duplicated().sum())

print("\nMissing values:")
print(
    catalogue_df[
        ["overview", "genres", "keywords", "cast", "director"]
    ].isna().sum()
)

print("\nMedia types:")
print(catalogue_df["media_type"].value_counts())

print("\nDataset shape:")
print(catalogue_df.shape)

Duplicate titles: 9

Missing values:
overview    0
genres      0
keywords    0
cast        0
director    0
dtype: int64

Media types:
media_type
tv       439
movie    421
Name: count, dtype: int64

Dataset shape:
(860, 18)


In [30]:
import ast
import re

In [31]:
text_columns = [
    "overview",
    "genres",
    "keywords",
    "cast",
    "director"
]

for col in text_columns:
    catalogue_df[col] = catalogue_df[col].fillna("")

In [32]:
def clean_text(value):
    
    if isinstance(value, list):
        value = " ".join(map(str, value))
    
    value = str(value)
    
    value = value.lower()
    
    value = re.sub(r"[^a-zA-Z0-9\s]", " ", value)
    
    value = re.sub(r"\s+", " ", value).strip()
    
    return value

In [33]:
for col in text_columns:
    catalogue_df[col] = catalogue_df[col].apply(clean_text)

In [34]:
catalogue_df["combined_text"] = (
    catalogue_df["overview"] + " " +
    catalogue_df["genres"] + " " +
    catalogue_df["keywords"] + " " +
    catalogue_df["cast"] + " " +
    catalogue_df["director"]
)

In [35]:
print(
    catalogue_df[
        ["title", "combined_text"]
    ].iloc[0]
)

title                                              The Maze Runner
combined_text    a teenager with no memory of his past finds hi...
Name: 0, dtype: str


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(
    catalogue_df["combined_text"]
)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (860, 10000)


In [37]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:", similarity_matrix.shape)

Similarity Matrix Shape: (860, 860)


In [38]:
def recommend(title, top_n=10):
    
    # Find the movie/show
    matches = catalogue_df[
        catalogue_df["title"].str.lower() == title.lower()
    ]
    
    if matches.empty:
        print("❌ Title not found.")
        return pd.DataFrame()
    
    # Get its index
    index = matches.index[0]
    
    # Get similarity scores
    scores = list(enumerate(similarity_matrix[index]))
    
    # Sort from highest to lowest
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    recommendations = []
    
    # Skip the title itself
    for movie_index, score in scores[1:top_n + 1]:
        
        recommendations.append({
            "title": catalogue_df.iloc[movie_index]["title"],
            "media_type": catalogue_df.iloc[movie_index]["media_type"],
            "similarity_score": round(score * 100, 2)
        })
    
    return pd.DataFrame(recommendations)

In [39]:
recommend(
    "Spider-Man: Brand New Day",
    top_n=10
)

,title,media_type,similarity_score
0,Spider-Man: No Way Home,movie,38.87
1,Spider-Man: Homecoming,movie,34.95
2,Spider-Man: Far From Home,movie,32.18
3,Spider-Man: Across the Spider-Verse,movie,25.58
4,The Amazing Spider-Man 2,movie,24.45
5,Spider-Man,movie,23.67
6,The Amazing Spider-Man,movie,20.50
7,Spider-Man 2,movie,17.79
8,Thunderbolts*,movie,16.39
9,Thor: Ragnarok,movie,15.86


In [40]:
def recommend_for_user(liked_titles, top_n=10):
    
    liked_indices = []
    
    for title in liked_titles:
        
        matches = catalogue_df[
            catalogue_df["title"].str.lower() == title.lower()
        ]
        
        if matches.empty:
            print(f"⚠️ Title not found: {title}")
        else:
            liked_indices.append(matches.index[0])
    
    
    if not liked_indices:
        print("❌ No valid titles found.")
        return pd.DataFrame()
    
    
    # Get TF-IDF vectors of liked titles
    liked_vectors = tfidf_matrix[liked_indices]
    
    # Create user's taste vector
    user_vector = liked_vectors.mean(axis=0)
    
    
    # Compare user taste with every title
    user_similarity = cosine_similarity(
        user_vector,
        tfidf_matrix
    ).flatten()
    
    
    # Rank titles
    ranked_indices = user_similarity.argsort()[::-1]
    
    
    recommendations = []
    
    for index in ranked_indices:
        
        # Don't recommend something the user already likes
        if index in liked_indices:
            continue
        
        recommendations.append({
            "title": catalogue_df.iloc[index]["title"],
            "media_type": catalogue_df.iloc[index]["media_type"],
            "similarity_score": round(
                user_similarity[index] * 100,
                2
            )
        })
        
        if len(recommendations) == top_n:
            break
    
    
    return pd.DataFrame(recommendations)

In [42]:
def recommend_for_user(liked_titles, top_n=10):

    liked_indices = []

    for title in liked_titles:

        matches = catalogue_df[
            catalogue_df["title"].str.lower() == title.lower()
        ]

        if matches.empty:
            print(f"⚠️ Title not found: {title}")
        else:
            liked_indices.append(matches.index[0])

    if not liked_indices:
        print("❌ No valid titles found.")
        return pd.DataFrame()

    # Get TF-IDF vectors of liked titles
    liked_vectors = tfidf_matrix[liked_indices]

    # Create user taste vector
    user_vector = np.asarray(
        liked_vectors.mean(axis=0)
    )

    # Compare user taste with every title
    user_similarity = cosine_similarity(
        user_vector,
        tfidf_matrix
    ).flatten()

    # Rank titles
    ranked_indices = user_similarity.argsort()[::-1]

    recommendations = []

    for index in ranked_indices:

        # Don't recommend titles already liked
        if index in liked_indices:
            continue

        recommendations.append({
            "title": catalogue_df.iloc[index]["title"],
            "media_type": catalogue_df.iloc[index]["media_type"],
            "similarity_score": round(
                user_similarity[index] * 100,
                2
            )
        })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)

In [44]:
import numpy as np 
recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=10
)

,title,media_type,similarity_score
0,Spider-Man: Far From Home,movie,51.25
1,Spider-Man: Across the Spider-Verse,movie,35.04
2,The Amazing Spider-Man 2,movie,31.54
3,Spider-Man,movie,29.34
4,The Amazing Spider-Man,movie,28.94
5,Spider-Man 2,movie,26.04
6,Doctor Strange in the Multiverse of Madness,movie,21.92
7,Iron Man 2,movie,21.36
8,Iron Man 3,movie,21.25
9,Avengers: Infinity War,movie,20.97


In [45]:
baseline_recommendations = recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=20
)

baseline_recommendations

,title,media_type,similarity_score
0,Spider-Man: Far From Home,movie,51.25
1,Spider-Man: Across the Spider-Verse,movie,35.04
2,The Amazing Spider-Man 2,movie,31.54
3,Spider-Man,movie,29.34
4,The Amazing Spider-Man,movie,28.94
5,Spider-Man 2,movie,26.04
6,Doctor Strange in the Multiverse of Madness,movie,21.92
7,Iron Man 2,movie,21.36
8,Iron Man 3,movie,21.25
9,Avengers: Infinity War,movie,20.97


In [46]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

catalogue_df[
    ["rating_norm", "popularity_norm"]
] = scaler.fit_transform(
    catalogue_df[
        ["vote_average", "popularity"]
    ]
)

print(
    catalogue_df[
        ["title", "vote_average", "rating_norm", "popularity", "popularity_norm"]
    ].head()
)

             title  vote_average  rating_norm  popularity  popularity_norm
0  The Maze Runner         7.190     0.625042     30.7066         0.013568
1     Forrest Gump         8.500     0.845804     35.5787         0.018214
2      Ratatouille         7.846     0.735592     31.2608         0.014096
3  Graphic Desires         7.000     0.593023     38.3307         0.020839
4           Kraken         6.325     0.479272     94.4635         0.074374


In [47]:
def hybrid_recommend_for_user(liked_titles, top_n=10):

    liked_indices = []

    for title in liked_titles:

        matches = catalogue_df[
            catalogue_df["title"].str.lower() == title.lower()
        ]

        if matches.empty:
            print(f"⚠️ Title not found: {title}")
        else:
            liked_indices.append(matches.index[0])

    if not liked_indices:
        print("❌ No valid titles found.")
        return pd.DataFrame()

    # User taste vector
    liked_vectors = tfidf_matrix[liked_indices]

    user_vector = np.asarray(
        liked_vectors.mean(axis=0)
    )

    # Content similarity
    content_scores = cosine_similarity(
        user_vector,
        tfidf_matrix
    ).flatten()

    # Hybrid score
    final_scores = (
        0.70 * content_scores
        + 0.20 * catalogue_df["rating_norm"].values
        + 0.10 * catalogue_df["popularity_norm"].values
    )

    # Rank
    ranked_indices = final_scores.argsort()[::-1]

    recommendations = []

    for index in ranked_indices:

        # Don't recommend already liked titles
        if index in liked_indices:
            continue

        recommendations.append({
            "title": catalogue_df.iloc[index]["title"],
            "media_type": catalogue_df.iloc[index]["media_type"],
            "content_score": round(
                content_scores[index] * 100, 2
            ),
            "rating": catalogue_df.iloc[index]["vote_average"],
            "final_score": round(
                final_scores[index] * 100, 2
            )
        })

        if len(recommendations) == top_n:
            break

    return pd.DataFrame(recommendations)

In [48]:
hybrid_recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=10
)

,title,media_type,content_score,rating,final_score
0,Spider-Man: Far From Home,movie,51.25,7.394,49.95
1,Spider-Man: Across the Spider-Verse,movie,35.04,8.341,41.41
2,Spider-Man,movie,29.34,7.342,34.93
3,The Amazing Spider-Man 2,movie,31.54,6.553,33.07
4,The Amazing Spider-Man,movie,28.94,6.749,32.06
5,Spider-Man 2,movie,26.04,7.324,31.37
6,Avengers: Infinity War,movie,20.97,8.200,31.31
7,The Avengers,movie,20.54,8.065,30.45
8,Marvel's Daredevil,tv,19.04,8.172,29.49
9,INVINCIBLE,tv,16.03,8.637,29.14


In [2]:
from sentence_transformers import SentenceTransformer

e:\ML\Movie\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Embedding model loaded")

e:\ML\Movie\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anike\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3573.07it/s]


✅ Embedding model loaded


In [10]:
import pandas as pd

catalogue_df = pd.read_csv(
    "../data/processed/movies_421_enriched.csv"
)

print("Dataset shape:", catalogue_df.shape)

Dataset shape: (421, 16)


In [12]:
import os

print(os.listdir("../data/processed"))

['movies_421_enriched.csv', 'movies_421_enriched.json', 'movies_enriched.csv', 'movies_enriched.json', 'movie_tv_catalogue.csv', 'movie_tv_catalogue.json']


In [14]:
catalogue_df = pd.read_csv(
    "../data/processed/movie_tv_catalogue.csv"
)

print("Dataset shape:", catalogue_df.shape)

Dataset shape: (860, 18)


In [15]:
print(catalogue_df.columns.tolist())

['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'director', 'runtime', 'release_date', 'vote_average', 'vote_count', 'popularity', 'poster_path', 'backdrop_path', 'original_language', 'media_type', 'number_of_seasons', 'number_of_episodes']


In [16]:
catalogue_df["combined_text"] = (
    catalogue_df["overview"].fillna("") + " " +
    catalogue_df["genres"].fillna("") + " " +
    catalogue_df["keywords"].fillna("") + " " +
    catalogue_df["cast"].fillna("") + " " +
    catalogue_df["director"].fillna("")
)

print(
    catalogue_df[["title", "combined_text"]].iloc[0]
)

title                                              The Maze Runner
combined_text    A teenager with no memory of his past finds hi...
Name: 0, dtype: str


In [17]:
print("Missing combined_text:",
      catalogue_df["combined_text"].isna().sum())

print("Total records:",
      len(catalogue_df))

Missing combined_text: 0
Total records: 860


In [18]:
semantic_embeddings = embedding_model.encode(
    catalogue_df["combined_text"].tolist(),
    show_progress_bar=True
)

print("Embedding Shape:", semantic_embeddings.shape)

Batches: 100%|██████████| 27/27 [01:00<00:00,  2.24s/it]

Embedding Shape: (860, 384)


In [19]:
print("Vector length:", len(semantic_embeddings[0]))

Vector length: 384


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def semantic_recommend_for_user(liked_titles, top_n=10):

    liked_indices = []

    for title in liked_titles:
        matches = catalogue_df.index[
            catalogue_df["title"].str.lower() == title.lower()
        ].tolist()

        if matches:
            liked_indices.append(matches[0])
        else:
            print(f"Movie not found: {title}")

    if not liked_indices:
        print("No valid movies found.")
        return pd.DataFrame()

    # Get embeddings of liked movies
    liked_vectors = semantic_embeddings[liked_indices]

    # Create user preference vector
    user_vector = liked_vectors.mean(axis=0).reshape(1, -1)

    # Compare with every movie
    similarities = cosine_similarity(
        user_vector,
        semantic_embeddings
    ).flatten()

    # Don't recommend movies already liked
    similarities[liked_indices] = -1

    # Get top recommendations
    top_indices = np.argsort(similarities)[::-1][:top_n]

    recommendations = catalogue_df.iloc[top_indices][
        ["title", "media_type"]
    ].copy()

    recommendations["semantic_score"] = (
        similarities[top_indices] * 100
    ).round(2)

    recommendations.reset_index(drop=True, inplace=True)

    return recommendations

In [21]:
semantic_recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=10
)

,title,media_type,semantic_score
0,The Amazing Spider-Man,movie,89.430000
1,Spider-Man 2,movie,86.419998
2,Spider-Man: Far From Home,movie,86.260002
3,Spider-Man,movie,85.440002
4,The Amazing Spider-Man 2,movie,85.279999
5,Spider-Man 3,movie,83.839996
6,Ant-Man and the Wasp,movie,83.669998
7,The Punisher: One Last Kill,movie,81.260002
8,Guardians of the Galaxy Vol. 3,movie,80.820000
9,Ant-Man,movie,80.760002


In [22]:
semantic_results = semantic_recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=10
)

semantic_results

,title,media_type,semantic_score
0,The Amazing Spider-Man,movie,89.430000
1,Spider-Man 2,movie,86.419998
2,Spider-Man: Far From Home,movie,86.260002
3,Spider-Man,movie,85.440002
4,The Amazing Spider-Man 2,movie,85.279999
5,Spider-Man 3,movie,83.839996
6,Ant-Man and the Wasp,movie,83.669998
7,The Punisher: One Last Kill,movie,81.260002
8,Guardians of the Galaxy Vol. 3,movie,80.820000
9,Ant-Man,movie,80.760002


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf.fit_transform(
    catalogue_df["combined_text"]
)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (860, 10000)


In [26]:
print("Catalogue:", catalogue_df.shape)
print("TF-IDF:", tfidf_matrix.shape)
print("Semantic:", semantic_embeddings.shape)

Catalogue: (860, 19)
TF-IDF: (860, 10000)
Semantic: (860, 384)


In [27]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd


def hybrid_recommend_for_user(liked_titles, top_n=10):

    liked_indices = []

    # Find liked titles
    for title in liked_titles:

        matches = catalogue_df.index[
            catalogue_df["title"].str.lower() == title.lower()
        ].tolist()

        if matches:
            liked_indices.append(matches[0])
        else:
            print(f"Movie not found: {title}")

    if not liked_indices:
        print("No valid titles found.")
        return pd.DataFrame()

    # ------------------------------------------------
    # 1. TF-IDF CONTENT SCORE
    # ------------------------------------------------

    liked_tfidf = tfidf_matrix[liked_indices]

    user_tfidf = np.asarray(
        liked_tfidf.mean(axis=0)
    )

    content_scores = cosine_similarity(
        user_tfidf,
        tfidf_matrix
    ).flatten()


    # ------------------------------------------------
    # 2. SEMANTIC SCORE
    # ------------------------------------------------

    liked_semantic = semantic_embeddings[liked_indices]

    user_semantic = (
        liked_semantic
        .mean(axis=0)
        .reshape(1, -1)
    )

    semantic_scores = cosine_similarity(
        user_semantic,
        semantic_embeddings
    ).flatten()


    # ------------------------------------------------
    # 3. NORMALIZE SCORES
    # ------------------------------------------------

    content_scores = (
        content_scores / content_scores.max()
    )

    semantic_scores = (
        semantic_scores / semantic_scores.max()
    )


    # ------------------------------------------------
    # 4. RATING SCORE
    # ------------------------------------------------

    rating_scores = (
        catalogue_df["vote_average"]
        .fillna(0)
        .values / 10
    )


    # ------------------------------------------------
    # 5. FINAL HYBRID SCORE
    # ------------------------------------------------

    final_scores = (
        0.35 * content_scores +
        0.50 * semantic_scores +
        0.15 * rating_scores
    )


    # ------------------------------------------------
    # 6. REMOVE ALREADY LIKED TITLES
    # ------------------------------------------------

    final_scores[liked_indices] = -1


    # ------------------------------------------------
    # 7. TOP RECOMMENDATIONS
    # ------------------------------------------------

    top_indices = np.argsort(
        final_scores
    )[::-1][:top_n]


    recommendations = catalogue_df.iloc[
        top_indices
    ][
        ["title", "media_type", "vote_average"]
    ].copy()


    recommendations["content_score"] = (
        content_scores[top_indices] * 100
    ).round(2)

    recommendations["semantic_score"] = (
        semantic_scores[top_indices] * 100
    ).round(2)

    recommendations["final_score"] = (
        final_scores[top_indices] * 100
    ).round(2)


    recommendations.reset_index(
        drop=True,
        inplace=True
    )

    return recommendations

In [28]:
hybrid_results = hybrid_recommend_for_user(
    [
        "Spider-Man: Brand New Day",
        "Spider-Man: No Way Home",
        "Spider-Man: Homecoming"
    ],
    top_n=10
)

hybrid_results

,title,media_type,vote_average,content_score,semantic_score,final_score
0,Spider-Man: Far From Home,movie,7.394,60.00,90.230003,77.20
1,The Amazing Spider-Man,movie,6.749,40.75,93.550003,71.16
2,Spider-Man: Across the Spider-Verse,movie,8.341,47.46,83.339996,70.79
3,Spider-Man,movie,7.342,37.26,89.370003,68.74
4,The Amazing Spider-Man 2,movie,6.553,39.92,89.199997,68.40
5,Spider-Man 2,movie,7.324,34.11,90.389999,68.12
6,The Avengers,movie,8.065,27.55,82.199997,62.84
7,Spider-Man 3,movie,6.472,26.11,87.699997,62.69
8,The Punisher: One Last Kill,movie,8.297,21.87,84.989998,62.60
9,Iron Man,movie,7.663,26.71,83.010002,62.34


In [29]:
import pickle

with open("../data/processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("../data/processed/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("../data/processed/semantic_embeddings.pkl", "wb") as f:
    pickle.dump(semantic_embeddings, f)

catalogue_df.to_pickle(
    "../data/processed/catalogue_df.pkl"
)

print("✅ ML components saved")

✅ ML components saved
